In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectFromModel,
    SelectKBest,
    f_classif
)

warnings.filterwarnings("ignore")

USE_MANUAL_5050_TEST = False

FILE_PATH = Path("/Users/francesco/Tesi/BC-ML4/dataset/cleaned")
df = pd.read_csv(FILE_PATH / "ambl_lesions.csv")

df["PR_class"] = (pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1).astype(int)
df = df.dropna(subset=["PR_class"])

drop_cols = [
    "Patient ID","lesion idx","tumor/benign",
    "GRADE","isTN","Breast",
    "ER [SII]","PR [SII]","HER2 [SII]",
    "PR_class"
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["PR_class"]

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean())

print("Totale campioni:", len(y))
print(y.value_counts())


Totale campioni: 82
PR_class
0    46
1    36
Name: count, dtype: int64


In [9]:
def create_manual_5050_splits(X, y, n_splits=5, random_state=42):
    np.random.seed(random_state)
    idx0 = y[y==0].index.to_numpy()
    idx1 = y[y==1].index.to_numpy()

    np.random.shuffle(idx0)
    np.random.shuffle(idx1)

    n_min = min(len(idx0), len(idx1))
    n_test = n_min // n_splits

    splits=[]
    for k in range(n_splits):
        test_idx = np.concatenate([
            idx0[k*n_test:(k+1)*n_test],
            idx1[k*n_test:(k+1)*n_test]
        ])
        train_idx = np.setdiff1d(np.arange(len(y)), test_idx)
        splits.append((train_idx,test_idx))
    return splits

if USE_MANUAL_5050_TEST:
    cv_splits = create_manual_5050_splits(X,y)
else:
    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    cv_splits = list(skf.split(X,y))


In [10]:
def run_cv_experiment(X, y, cv_splits, selector=None, name="BASELINE"):

    print(f"\n================ {name} ================\n")

    accs, bals, f1s, aucs = [], [], [], []

    for fold,(tr,te) in enumerate(cv_splits,1):

        X_train, X_test = X.iloc[tr], X.iloc[te]
        y_train, y_test = y.iloc[tr], y.iloc[te]

        n_neg = (y_train==0).sum()
        n_pos = (y_train==1).sum()
        scale_pos_weight = n_neg/n_pos

        # ===== FEATURE SELECTION =====
        if selector is not None:
            X_train = selector.fit_transform(X_train, y_train)
            X_test  = selector.transform(X_test)
            print(f"FOLD {fold}: feature dopo selezione = {X_train.shape[1]}")
        else:
            print(f"FOLD {fold}: feature = {X_train.shape[1]}")

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.6,
            min_child_weight=1,
            reg_alpha=0.5,
            reg_lambda=1,
            random_state=42,
            n_jobs=1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        # ===== METRICHE =====
        acc = accuracy_score(y_test, y_pred)
        bal = balanced_accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        accs.append(acc)
        bals.append(bal)
        f1s.append(f1)
        aucs.append(auc)

        # ===== STAMPA FOLD =====
        print(f"\n{'='*50}")
        print(f"FOLD {fold}")
        print(f"{'='*50}")
        print(f"TEST SET - Classe 0: {(y_test==0).sum()} | Classe 1: {(y_test==1).sum()}")
        print(f"Accuracy: {acc:.4f} | Balanced Accuracy: {bal:.4f}")
        
        # ===== CLASSIFICATION REPORT =====
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, zero_division=0))
        
        # ===== CONFUSION MATRIX =====
        cm = confusion_matrix(y_test, y_pred)
        print("Confusion Matrix:")
        print(cm)
        
        tn, fp, fn, tp = cm.ravel()
        print(f"TN={tn} | FP={fp} | FN={fn} | TP={tp}")

    # ===== RISULTATI FINALI =====
    print("\n" + "="*60)
    print(f"RISULTATI FINALI - {name}")
    print("="*60)
    print(f"Accuracy:         {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"Balanced Accuracy: {np.mean(bals):.3f} ± {np.std(bals):.3f}")
    print(f"F1-score:         {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"ROC-AUC:          {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

In [11]:
run_cv_experiment(X,y,cv_splits,name="BASELINE")



================ BASELINE ================

FOLD 1: feature = 105

FOLD 1
TEST SET - Classe 0: 10 | Classe 1: 7
Accuracy: 0.5294 | Balanced Accuracy: 0.5143

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.60      0.60        10
           1       0.43      0.43      0.43         7

    accuracy                           0.53        17
   macro avg       0.51      0.51      0.51        17
weighted avg       0.53      0.53      0.53        17

Confusion Matrix:
[[6 4]
 [4 3]]
TN=6 | FP=4 | FN=4 | TP=3
FOLD 2: feature = 105

FOLD 2
TEST SET - Classe 0: 9 | Classe 1: 8
Accuracy: 0.5294 | Balanced Accuracy: 0.5139

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.78      0.64         9
           1       0.50      0.25      0.33         8

    accuracy                           0.53        17
   macro avg       0.52      0.51      0.48        17
weighted avg       0.52 

In [12]:
vt = VarianceThreshold(threshold=1e-4)
run_cv_experiment(X,y,cv_splits,selector=vt,name="VarianceThreshold")



================ VarianceThreshold ================

FOLD 1: feature dopo selezione = 103

FOLD 1
TEST SET - Classe 0: 10 | Classe 1: 7
Accuracy: 0.5294 | Balanced Accuracy: 0.4929

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.70      0.64        10
           1       0.40      0.29      0.33         7

    accuracy                           0.53        17
   macro avg       0.49      0.49      0.48        17
weighted avg       0.51      0.53      0.51        17

Confusion Matrix:
[[7 3]
 [5 2]]
TN=7 | FP=3 | FN=5 | TP=2
FOLD 2: feature dopo selezione = 103

FOLD 2
TEST SET - Classe 0: 9 | Classe 1: 8
Accuracy: 0.5294 | Balanced Accuracy: 0.5139

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.78      0.64         9
           1       0.50      0.25      0.33         8

    accuracy                           0.53        17
   macro avg       0.52      0.51      

In [13]:
sfm_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    n_jobs=1
)

sfm = SelectFromModel(sfm_model, threshold="median")
run_cv_experiment(X,y,cv_splits,selector=sfm,name="SelectFromModel")



================ SelectFromModel ================

FOLD 1: feature dopo selezione = 53

FOLD 1
TEST SET - Classe 0: 10 | Classe 1: 7
Accuracy: 0.5294 | Balanced Accuracy: 0.4929

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.70      0.64        10
           1       0.40      0.29      0.33         7

    accuracy                           0.53        17
   macro avg       0.49      0.49      0.48        17
weighted avg       0.51      0.53      0.51        17

Confusion Matrix:
[[7 3]
 [5 2]]
TN=7 | FP=3 | FN=5 | TP=2
FOLD 2: feature dopo selezione = 53

FOLD 2
TEST SET - Classe 0: 9 | Classe 1: 8
Accuracy: 0.5294 | Balanced Accuracy: 0.5139

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.78      0.64         9
           1       0.50      0.25      0.33         8

    accuracy                           0.53        17
   macro avg       0.52      0.51      0.48

In [14]:
skb = SelectKBest(f_classif, k=30)
run_cv_experiment(X,y,cv_splits,selector=skb,name="SelectKBest(k=30)")



================ SelectKBest(k=30) ================

FOLD 1: feature dopo selezione = 30

FOLD 1
TEST SET - Classe 0: 10 | Classe 1: 7
Accuracy: 0.7059 | Balanced Accuracy: 0.7071

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.70      0.74        10
           1       0.62      0.71      0.67         7

    accuracy                           0.71        17
   macro avg       0.70      0.71      0.70        17
weighted avg       0.71      0.71      0.71        17

Confusion Matrix:
[[7 3]
 [2 5]]
TN=7 | FP=3 | FN=2 | TP=5
FOLD 2: feature dopo selezione = 30

FOLD 2
TEST SET - Classe 0: 9 | Classe 1: 8
Accuracy: 0.5882 | Balanced Accuracy: 0.5764

Classification Report:
              precision    recall  f1-score   support

           0       0.58      0.78      0.67         9
           1       0.60      0.38      0.46         8

    accuracy                           0.59        17
   macro avg       0.59      0.58      0.